In [ ]:
# !pip install langchain langchain-community langchain-core langchain-text-splitters langchain-ollama faiss-cpu sentence-transformers pypdf



### RAG-Based Question Answering from PDF using LangChain, FAISS & Ollama”


In [3]:

from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import FAISS

from langchain_community.docstore.in_memory import InMemoryDocstore

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_ollama import ChatOllama

from langchain_core.prompts import ChatPromptTemplate

import faiss


import warnings
warnings.filterwarnings("ignore")

In [4]:
#   Load and split PDF 

FILE_PATH = "usa.pdf"

loader = PyPDFLoader(FILE_PATH)

pages = loader.load()

print(f" Loaded {len(pages)} pages")


 Loaded 3 pages


In [5]:

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)

split_docs = splitter.split_documents(pages)

print(f" Split into {len(split_docs)} chunks")


 Split into 12 chunks


In [6]:
# ===  Create embeddings & FAISS index ===

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
embeddings

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 513.98it/s]


HuggingFaceEmbeddings(model_name='sentence-transformers/all-mpnet-base-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [7]:

embedding_dim = 768

In [8]:

index = faiss.IndexFlatL2(embedding_dim)


In [9]:

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={}
)

vector_store.add_documents(split_docs)
print(" Documents embedded and indexed")


 Documents embedded and indexed


In [10]:

retriever = vector_store.as_retriever(search_kwargs={"k": 4})
retriever


VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001D3D4F11BB0>, search_kwargs={'k': 4})

In [16]:
# ===  Create LLM ===

llm = ChatOllama(model="llama3.2:latest")


In [17]:

# ===  Build a custom retrieval + prompt chain ===
query = "What is the USA GDP in 2024?"


In [18]:
# Retrieve documents

retrieved_docs = retriever.invoke(query)

context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])


In [19]:
# Create prompt template

prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant. Give me factual answer from given context only.

Use the below context to answer the user's query.

Context:
{context}

Question:
{question}

Answer clearly and concisely.
""")

# Format prompt
formatted_prompt = prompt.format(context=context_text, question=query)
formatted_prompt

"Human: \nYou are a helpful assistant. Give me factual answer from given context only.\n\nUse the below context to answer the user's query.\n\nContext:\nand 2023 saw rising inflation due to supply chain issues and stimulus-driven demand. \nIn 2024, the U.S. GDP grew at a modest pace of around 2.1%, as the Federal Reserve’s \ninterest rate hikes aimed at controlling inflation also moderated economic expansion. \nConsumer spending, which makes up nearly 70% of GDP , remains a dominant force in \neconomic stability.\n\nreal estate, defense, and agriculture. \n \nU.S. GDP – Size, Composition, and Global Share \nAs of 2024, the United States’ nominal GDP is estimated to be around $28 trillion USD, \naccounting for approximately 25% of the global economy. It ranks #1 in the world by \nnominal GDP , far ahead of China (which ranks 2nd). The U.S. GDP per capita is also \namong the highest, hovering around $83,000, which indicates a high standard of living \nand productivity. In terms of Purcha

In [20]:
# Get response from Ollama

response = llm.invoke(formatted_prompt)

print("\n Answer:\n", response.content)


 Answer:
 The U.S. GDP in 2024 is estimated to be around $28 trillion USD.


In [21]:

print("\n Retrieved sources:\n")

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"--- Document {i} ---")
    print(f"Page: {doc.metadata.get('page', 'N/A')}")
    print(doc.page_content[:400], "...\n")



 Retrieved sources:

--- Document 1 ---
Page: 0
and 2023 saw rising inflation due to supply chain issues and stimulus-driven demand. 
In 2024, the U.S. GDP grew at a modest pace of around 2.1%, as the Federal Reserve’s 
interest rate hikes aimed at controlling inflation also moderated economic expansion. 
Consumer spending, which makes up nearly 70% of GDP , remains a dominant force in 
economic stability. ...

--- Document 2 ---
Page: 0
real estate, defense, and agriculture. 
 
U.S. GDP – Size, Composition, and Global Share 
As of 2024, the United States’ nominal GDP is estimated to be around $28 trillion USD, 
accounting for approximately 25% of the global economy. It ranks #1 in the world by 
nominal GDP , far ahead of China (which ranks 2nd). The U.S. GDP per capita is also 
among the highest, hovering around $83,000, which in ...

--- Document 3 ---
Page: 0
77% to the total GDP . This includes industries such as finance, insurance, real estate, 
healthcare, education, and informa